In [1]:
%%configure -f
{
    "conf": {
        "spark.jars": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar ,s3://ne-prod-delta-lake-secure-layer/elasticmapreduce/external-jars/encrypt-decrypt-spark-1.0-SNAPSHOT.jar",
        "spark.submit.pyFiles": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar,s3://ne-prod-data-pipeline-orchestrator/ds-da-repo/emr_spark_utils.zip"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1781603845101_0001,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_shramana_patra_slicebank_com,


In [2]:
from decimal import Decimal
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import openpyxl as xl
from openpyxl.utils import get_column_interval
from openpyxl.utils.dataframe import dataframe_to_rows
import re
import json
import operator
import boto3
import email
import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import time
import enum
from pyspark.sql import SparkSession
import math
import io
from pyspark.sql import functions as F
import pytz
import requests

from pyspark.sql.functions import col, split, get_json_object, date_sub, current_timestamp, max as spark_max, min as spark_min, collect_list, slice
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import col, split, get_json_object, current_date, date_sub, year, month, dayofmonth, broadcast, expr, sort_array, struct
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import explode, row_number

from pyspark.sql.window import Window
# Compute yesterday and today
yesterday = date_sub(current_date(), 1)
today = current_date()


s3 = boto3.client('s3')
date_yesterday = datetime.now() - timedelta(days=1)

date_yesterday_str = date_yesterday.strftime('%Y-%m-%d')

def get_secrets(secret_name):
    region_name = "ap-south-1"
    session = boto3.session.Session()

    client = session.client(service_name="secretsmanager", region_name=region_name)

    try:
        get_secret_value_response = client.get_secret_value(SecretId=secret_name)

    except ClientError as e:
        if e.response.get("Error").get("Code") == "ResourceNotFoundException":
            raise Exception("The requested secret " + secret_name + " was not found")
        elif e.response.get("Error").get("Code") == "InvalidRequestException":
            raise Exception("The request was invalid due to:", e)
        else:
            raise Exception("The request failed because of:", e)

    secret = get_secret_value_response.get("SecretString")

    if isinstance(secret, str):
        secret = eval(secret)

    return secret

MSSQL_ANALYTICS_CLUSTER_COMMON = "prod/data/analytics/cbs/common" #verified
BANKOS = "prod/data/analytics/bankos"
MIS_8004 = "prod/data/analytics/cbs/mis_8004"
MIS_2397 = "prod/data/analytics/cbs/mis_2397"
BANK_KYC = "prod/data/analytics/bank_kyc"
BANK_KYC_RW = "prod/data/analytics/bankkyc_rw"
BANK_ONBOARDING_DB = "prod/data/analytics/bank_onboarding"
BANK_ONBOARDING_DB_RW = "prod/data/analytics/bank_onboarding_rw"
COMMON_TOKEN = 'prod/data/analytics/common-token'
SUNRISE_MAIL = 'prod/data/analytics/sunrise-mail'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V2  = 'prod/data/analytics/bank-payment-links-access-token-v2'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V3  = 'prod/data/analytics/bank-payment-links-access-token-v3'
RAZORPAY = 'prod/data/analytics/cbs/razorpay'
HELO_AI = 'prod/data/analytics/heloai'
SERVICE_TOKEN = 'prod/data/analytics/service-token'
CONFLUENT_TOKEN = 'prod/data/analytics/confluent-token'

    
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
from datetime import datetime
slack_token =get_secrets(COMMON_TOKEN)["slack_token"]
client = WebClient(token=slack_token)

spark.udf.registerJavaFunction("decryptFunction", "com.piixie.SparkDecrypt")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

def push_to_slack(df1):
    channel_id = "C0958LSLM8Q"
    df1.to_csv('./sample.csv', index=False)
    if df1.shape[0] > 0 : 
        client.files_upload_v2(
        channel = channel_id,
        title = "df_base",
        file = "./sample.csv",
        initial_comment="df_base",
        )
        print(f'pushed to slack: {channel_id}')

def load_from_s3(path, view_name):
    data = spark.read.parquet(path)
    data.createOrReplaceTempView(view_name)
    
    return data
    
def load_csv(path, view_name):    
    import pandas as pd
    import io
    import boto3

    # Parse S3 path
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    data = io.BytesIO(obj['Body'].read())
    df = pd.read_csv(data)

    # Convert to Spark DataFrame
    df = spark.createDataFrame(df)

    # Create temporary view
    df.createOrReplaceTempView(view_name)

def load_excel(path, view_name):
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    excel_data = io.BytesIO(obj['Body'].read())
    df = pd.read_excel(excel_data)
    df.columns = df.columns.str.strip()

    # Convert to Spark DataFrame
    blocked_df = spark.createDataFrame(df)
    blocked_df.createOrReplaceTempView(view_name)
    
    return blocked_df

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

######### TOKEN SET RATIO

from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import IntegerType
import re
from difflib import SequenceMatcher

def tokenize(s):
    if s is None:
        return []
    s = re.sub(r'[^a-zA-Z0-9 ]', '', s.lower())
    return s.split()

def ratio(s1, s2):
    return int(SequenceMatcher(None, s1, s2).ratio() * 100)

def token_set_ratio(s1, s2):
    if s1 is None or s2 is None:
        return 0
    
    tokens1 = set(tokenize(s1))
    tokens2 = set(tokenize(s2))
    
    common_tokens = tokens1.intersection(tokens2)
    diff1 = tokens1.difference(tokens2)
    diff2 = tokens2.difference(tokens1)
    
    sorted_common = ' '.join(sorted(common_tokens))
    sorted_diff1 = ' '.join(sorted(diff1))
    sorted_diff2 = ' '.join(sorted(diff2))
    
    # Combine strings in three ways and calculate ratio
    combined1 = sorted_common + ' ' + sorted_diff1
    combined2 = sorted_common + ' ' + sorted_diff2
    
    ratios = [
        ratio(sorted_common, combined1),
        ratio(sorted_common, combined2),
        ratio(combined1, combined2),
    ]
    
    return max(ratios)


token_set_ratio_udf = udf(token_set_ratio, IntegerType())

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1781603845101_0002,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_rayansh_khamesra_slicebank_com,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
import pandas as pd

data = [
    # Payments Banks
    {"bank_name": "Airtel Payments Bank", "ifsc_prefix": "AIRP", "type": "payments"},
    {"bank_name": "Jio Payments Bank Ltd", "ifsc_prefix": "JIOP", "type": "payments"},
    {"bank_name": "Fino Payments Bank", "ifsc_prefix": "FINO", "type": "payments"},
    {"bank_name": "India Post Payments Bank", "ifsc_prefix": "IPOS", "type": "payments"},
    {"bank_name": "NSDL Payments Bank Ltd", "ifsc_prefix": "NSPB", "type": "payments"},

    # Gramin / RRB Banks
    {"bank_name":"Rajasthan Marudhara Gramin Bank","ifsc_prefix":"RMGB","type":"gramin"},
    {"bank_name":"Baroda Rajasthan Kshetriya Gramin Bank","ifsc_prefix":"BARB","type":"gramin"},
    {"bank_name":"Dakshin Bihar Gramin Bank","ifsc_prefix":"PUNB","type":"gramin"},
    {"bank_name":"Kerala Gramin Bank","ifsc_prefix":"KLGB","type":"gramin"},
    {"bank_name":"Telangana Grameena Bank","ifsc_prefix":"TGRB","type":"gramin"},
    {"bank_name":"Uttarakhand Gramin Bank","ifsc_prefix":"SBIN","type":"gramin"},
    {"bank_name":"Maharashtra Gramin Bank","ifsc_prefix":"MAHG","type":"gramin"},
    {"bank_name":"Tripura Gramin Bank","ifsc_prefix":"TRGB","type":"gramin"},
    {"bank_name":"Assam Gramin Vikash Bank","ifsc_prefix":"AGVB","type":"gramin"},
    {"bank_name":"Pragathi Krishna Gramin Bank","ifsc_prefix":"PKGB","type":"gramin"},
    {"bank_name":"West Bengal Gramin Bank","ifsc_prefix":"WBGB","type":"gramin"},
    {"bank_name":"Uttar Bihar Gramin Bank","ifsc_prefix":"CBIN","type":"gramin"},
    {"bank_name":"Paschim Banga Gramin Bank","ifsc_prefix":"PGBG","type":"gramin"},
    {"bank_name":"Vidharbha Konkan Gramin Bank","ifsc_prefix":"VKGB","type":"gramin"}
]

df = pd.DataFrame(data)

gp_banks = spark.createDataFrame(df)

gp_banks.createOrReplaceTempView('bank_type')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
%%pretty
spark.sql("""
SELECT 
    a.*, 
    CAST(b.onboarding_date AS DATE) AS onboarding_date,
    c.bank_name,
    c.ifsc_prefix,
    c.type
FROM cyber_crime_gold.final_view_deduped a
LEFT JOIN druid_gold.user_savings_details b ON a.uuid = b.user_id
LEFT JOIN bank_type c ON LEFT(a.counter_party_ifsc, 4) = c.ifsc_prefix
WHERE CAST(onboarding_date AS DATE) BETWEEN DATE '2026-06-01' AND DATE '2026-06-12'
AND b.account_type='CURRENT'
""").createOrReplaceTempView('txn_analysis_data')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
%%pretty
spark.sql("""
WITH running_totals AS (
    SELECT
        account_no,
        uuid,
        txn_ref_no,
        ifsc_prefix,
        txn_amount,
        txn_nature,
        SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'CBIN' THEN txn_amount ELSE 0 END) 
            OVER (PARTITION BY account_no ORDER BY transactiontime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cbin_running,
        SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'FINO' THEN txn_amount ELSE 0 END) 
            OVER (PARTITION BY account_no ORDER BY transactiontime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS fino_running,
        SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'JIOP' THEN txn_amount ELSE 0 END) 
            OVER (PARTITION BY account_no ORDER BY transactiontime ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS jiop_running
    FROM txn_analysis_data
    WHERE txn_mode <> 'IFT - Interest Credits'
        AND DATE(transactiontime) BETWEEN onboarding_date AND onboarding_date + INTERVAL '7' DAY
),
anchor_txns AS (
    SELECT
        account_no,
        -- First txn where ALL three banks cross 1000
        MIN(CASE WHEN cbin_running > 1000 AND fino_running > 1000 AND jiop_running > 1000 
            THEN txn_ref_no END) AS anchor_txn_ref_no
    FROM running_totals
    GROUP BY account_no
)
SELECT
    t.account_no,
    t.uuid,
    a.anchor_txn_ref_no,
    SUM(CASE WHEN txn_nature = 'C' THEN txn_amount ELSE 0 END) AS credit_amount,
    SUM(CASE WHEN txn_nature = 'D' THEN txn_amount ELSE 0 END) AS debit_amount,
    
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'CBIN' THEN txn_amount END) AS cbin_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'CBIN' THEN txn_amount END) AS cbin_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'CBIN' THEN counter_party_cbs_name END) AS cbin_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'JIOP' THEN txn_amount END) AS jiop_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'JIOP' THEN txn_amount END) AS jiop_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'JIOP' THEN counter_party_cbs_name END) AS jiop_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'FINO' THEN txn_amount END) AS fino_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'FINO' THEN txn_amount END) AS fino_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'FINO' THEN counter_party_cbs_name END) AS fino_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'RMGB' THEN txn_amount END) AS rmgb_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'RMGB' THEN txn_amount END) AS rmgb_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'RMGB' THEN counter_party_cbs_name END) AS rmgb_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'PKGB' THEN txn_amount END) AS pkgb_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'PKGB' THEN txn_amount END) AS pkgb_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'PKGB' THEN counter_party_cbs_name END) AS pkgb_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'MAHG' THEN txn_amount END) AS mahg_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'MAHG' THEN txn_amount END) AS mahg_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'MAHG' THEN counter_party_cbs_name END) AS mahg_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'TGRB' THEN txn_amount END) AS tgrb_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'TGRB' THEN txn_amount END) AS tgrb_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'TGRB' THEN counter_party_cbs_name END) AS tgrb_cp_count,
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'KLGB' THEN txn_amount END) AS klgb_credit_amount,
    COUNT(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'KLGB' THEN txn_amount END) AS klgb_credit_count,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'KLGB' THEN counter_party_cbs_name END) AS klgb_cp_count,
    
    SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix IN ('CBIN', 'JIOP', 'FINO', 'RMGB', 'PKGB', 'MAHG', 'TGRB', 'KLGB') THEN txn_amount ELSE 0 END) AS risky_bank_credit_amount,
    COUNT(DISTINCT CASE WHEN txn_nature = 'C' AND ifsc_prefix IN ('CBIN', 'JIOP', 'FINO', 'RMGB', 'PKGB', 'MAHG', 'TGRB', 'KLGB') THEN ifsc_prefix END) AS risky_bank_count

FROM txn_analysis_data t
INNER JOIN anchor_txns a ON t.account_no = a.account_no
WHERE t.txn_mode <> 'IFT - Interest Credits'
  AND DATE(t.transactiontime) BETWEEN t.onboarding_date AND t.onboarding_date + INTERVAL '7' DAY
GROUP BY 1, 2, 3
HAVING SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'CBIN' THEN txn_amount END) > 1000
   AND SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'FINO' THEN txn_amount END) > 1000
   AND SUM(CASE WHEN txn_nature = 'C' AND ifsc_prefix = 'JIOP' THEN txn_amount END) > 1000
""").show(10, 0)